## chatbot Evaluation

In [ ]:
## chatbot Evaluation
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"

from langsmith import Client
client = Client()

dataset_name = "Simple Chatbot Evaluation"

## Safer: try to read it first, only create if it truly doesn't exist
try:
    dataset = client.read_dataset(dataset_name=dataset_name)
    print(f"Found existing dataset: {dataset.id}")
except Exception:
    dataset = client.create_dataset(dataset_name)
    print(f"Created new dataset: {dataset.id}")

## Add examples (safe to re-run; LangSmith dedupes exact same examples on some SDK versions,
## but to be fully safe, check example count first)
existing = list(client.list_examples(dataset_id=dataset.id))
if len(existing) == 0:
    client.create_examples(
        dataset_id=dataset.id,
        inputs=[
            {"question": "What is Langchain?"},
            {"question": "What is langsmith"},
        ],
        outputs=[
            {"answer": "A framework for building llm applications"},
            {"answer": "a platform for observing and evaluation LLM applications"},
        ],
    )
    print("Added 2 examples")
else:
    print(f"Dataset already has {len(existing)} examples, skipping insert")

### Define Metric (LLM as a judge)

In [ ]:

from groq import Groq

groq_client = Groq()

instructions = "You are an exper professor in grading students answer to questions"

def correctness(inputs:dict,outputs:dict,reference_outputs:dict) -> bool:
    user_content = f""" You are grading the following questions:
    {inputs['question']}
Here is the real answer:
{reference_outputs['answer']}
You are grading the following predicted answer:
{outputs['response']}
Respond with CORRECT or INCORRECT:
Grade:
"""
    response = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_content},
    ],
    temperature=0,
  ).choices[0].message.content
   
    return response == "CORRECT"



In [ ]:
## Concisions

def concision(outputs:dict,reference_outputs:dict)->bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

### how to run evaluation


In [ ]:
default_instruction = "Respond to the users question in a short, concise manner(one short sentence)."
def my_app(question:str,model:str="llama-3.3-70b-versatile",instructions:str = default_instruction)->str:
    return groq_client.chat.completions.create(
        model = model,
        temperature=0,
        messages=[
            {"role":"system","content":instructions},
            {"role":"user","content":question},
        ]
    ).choices[0].message.content

In [ ]:
### Call my_app for every datapoints
def ls_target(inputs:str)->dict:
    return {"response":my_app(inputs["question"])}

In [ ]:
## Run our evaluation

experiment_results = client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="llama-3.3-70b-versatile"
)

## Evaluation for RAG

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/"
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

#Split
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500, chunk_overlap=0
)

doc_splits = text_splitter.split_documents(docs_list)

## Add to vectorstore
vectorstore = InMemoryVectorStore.from_documents(
    documents = doc_splits,
    embedding = HuggingFaceEmbeddings()
)

retriever = vectorstore.as_retriever(k=6)

In [ ]:
retriever.invoke("What is agents ?")

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    "groq:llama-3.3-70b-versatile",
    temperature=0.7,
)

llm.invoke("What is agents")

In [ ]:
from langsmith import traceable

# For rag_bot — plain chat model, NO "groq:" prefix if using ChatGroq directly,
# or keep "groq:" prefix ONLY if using init_chat_model
chat_llm = init_chat_model("groq:llama-3.3-70b-versatile", temperature=0.7)

@traceable
def rag_bot(question: str) -> dict:
    docs = retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)
    instructions = f"""...
Documents:
{docs_string}"""
    ai_msg = chat_llm.invoke([   # <-- uses chat_llm, not llm
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ])
    return {"answer": ai_msg.content, "documents": docs}
    


In [ ]:
rag_bot("What is agents")

### Test the dataset

In [ ]:
from langsmith import Client

client = Client()

examples = [
    {
        "inputs": {"question": "how does the ReAct agent use self reflection?"},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions such as tool use"},
    }
]

DATASET_NAME = "RAG TEST"

try:
    dataset = client.read_dataset(dataset_name=DATASET_NAME)
    print(f"Using existing dataset: {dataset.id}")
except Exception:
    dataset = client.create_dataset(dataset_name=DATASET_NAME)
    client.create_examples(
        dataset_id=dataset.id,
        examples=examples,
    )
    print(f"Created new dataset with {len(examples)} example(s): {dataset.id}")

## Evaluators

#Correctness

In [ ]:
from typing_extensions import Annotated,TypedDict


## Correctness Output schema

class CorrectnessGrade(TypedDict):
    explanation:Annotated[str,"Explain you reasoning for the score "]
    correct:Annotated[bool,"True if the answer is correct, False otherwise"]

    ## correctness prompt
correctness_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer.
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it does not contain any conflicting statements.

Correct = True means the student's answer meets all of the criteria.
Correct = False means the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.
Avoid simply stating the correct answer at the outset."""

from langchain_groq import ChatGroq

grader_llm = ChatGroq(model="groq:llama-3.3-70b-versatile",temperature=0).with_structured_output(CorrectnessGrade,method="json_schema",strict=True)



In [ ]:

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    answers = f"""QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}"""

    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers},
    ])
    return grade["correct"]

## Relevance 

In [ ]:
from typing_extensions import Annotated, TypedDict
from langchain_groq import ChatGroq

## Relevance Output schema

class RelevanceGrade(TypedDict):
    explanation: Annotated[str, "Explain your reasoning for the score"]
    relevant: Annotated[bool, "True if the answer is relevant to the question, False otherwise"]

relevance_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION and a STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION.
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION.

Relevant = True means the student's answer meets all of the criteria.
Relevant = False means the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.
Avoid simply stating the correct answer at the outset."""

relevance_grader = ChatGroq(model="llama-3.3-70b-versatile", temperature=0).with_structured_output(RelevanceGrade)

def relevance(inputs: dict, outputs: dict) -> bool:
    answer = f"""QUESTION: {inputs['question']}
STUDENT ANSWER: {outputs['answer']}"""

    grade = relevance_grader.invoke([
        {"role": "system", "content": relevance_instructions},
        {"role": "user", "content": answer},
    ])
    return grade["relevant"]

In [ ]:
from typing_extensions import Annotated, TypedDict
from langchain_groq import ChatGroq

## Groundedness Output schema

class GroundedGrade(TypedDict):
    explanation: Annotated[str, "Explain your reasoning for the score"]
    grounded: Annotated[bool, "True if the answer is grounded in the facts, False otherwise"]

grounded_instructions = """You are a teacher grading a quiz.

You will be given FACTS and a STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS.
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded = True means the student's answer meets all of the criteria.
Grounded = False means the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.
Avoid simply stating the correct answer at the outset."""

grounded_grader = ChatGroq(model="llama-3.3-70b-versatile", temperature=0).with_structured_output(GroundedGrade)

def groundedness(inputs: dict, outputs: dict) -> bool:
    doc_string = " ".join(doc.page_content for doc in outputs["documents"])
    answer = f"""FACTS: {doc_string}
STUDENT ANSWER: {outputs['answer']}"""

    grade = grounded_grader.invoke([
        {"role": "system", "content": grounded_instructions},
        {"role": "user", "content": answer},
    ])
    return grade["grounded"]

Retrieval Relevance

In [ ]:
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, "Explain your reasoning for the score"]
    relevant: Annotated[bool, "True if the retrieved documents are relevant to the question, False otherwise"]

retrieval_relevance_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION and a set of FACTS provided by the student.

Here is the grade criteria to follow:
(1) Your goal is to identify FACTS that are completely unrelated to the QUESTION.
(2) It is OK if the facts have SOME information that is unrelated to the question, as long as (1) is met.
(3) It is OK if some retrieved documents are irrelevant, as long as at least one relevant document was retrieved.

Relevant = True means the FACTS contain at least some information that would help answer the QUESTION.
Relevant = False means the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.
Avoid simply stating the correct answer at the outset."""

retrieval_relevance_grader = ChatGroq(model="llama-3.3-70b-versatile", temperature=0).with_structured_output(RetrievalRelevanceGrade)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    doc_string = " ".join(doc.page_content for doc in outputs["documents"])
    answer = f"""FACTS: {doc_string}
QUESTION: {inputs['question']}"""

    grade = retrieval_relevance_grader.invoke([
        {"role": "system", "content": retrieval_relevance_instructions},
        {"role": "user", "content": answer},
    ])
    return grade["relevant"]

## Run the Evaluation

In [ ]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=dataset.id,   # pass the ID directly, not dataset_name
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version": "LCEL context"},
)
experiment_results.to_pandas()